In [1]:
# Jupyter Notebook

# Importando as bibliotecas necessárias
import os
import json
import subprocess
import statsmodels
import pandas as pd
import seaborn as sns
from pathlib import Path
from matplotlib.patches import Patch
import numpy as np
import matplotlib.pyplot as plt
import EcoSimpy

# Configurações iniciais
sns.set_theme(style="whitegrid")
pd.options.mode.chained_assignment = None  # Para evitar warnings de cópias de DataFrame

# Configurando os diretórios
input_dir = Path("runs")
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)



## Execução da Simulação

In [5]:
# def main():
#     # Activate virtual environment (if needed)
#     venv_path = os.path.join(os.getcwd(), "venv")
#     if os.path.exists(venv_path):
#         activate_script = os.path.join(venv_path, "bin", "activate_this.py")
#         with open(activate_script) as f:
#             exec(f.read(), {"__file__": activate_script})

EcoSimpy.make_config_json("C:\\Users\\Nathan\\Home\\gEco\\ipd\\ipd")

config_json_file = "config.json"
scenario_file = "zero_det_scn_base_05.json"
model_file = "model_zero_det_base_05.json"

#app_dir = os.getcwd()
app_dir = "C:\\Users\\Nathan\\Home\\gEco\\ipd\\ipd\\"

new_sim = EcoSimpy.Simulation(app_dir,
                              config_json_file,
                              model_file,
                              scenario_file,
																														clean_run = True)

new_sim.execute_simulation()



config.json generated successfully at C:\Users\Nathan\Home\gEco\ipd\ipd\config.json


Scenario: ZD_Scenarios Run nr.: 4: 100%|██████████| 5000/5000 [00:31<00:00, 157.46it/s]


Observer data saved to ipd_GoodPlayer_obs1_2026-04-14T13_46_.csv
Observer data saved to ipd_BadPlayer_obs1_2026-04-14T13_46_.csv
Observer data saved to ipd_RandomPlayer_obs1_2026-04-14T13_46_.csv
Observer data saved to ipd_TitForTatWithRecallPlayer_obs1_2026-04-14T13_46_.csv
Observer data saved to ipd_RancorousPlayer_obs1_2026-04-14T13_46_.csv
Observer data saved to ipd_PeriodicPlayer_obs1_2026-04-14T13_46_.csv
Observer data saved to ipd_HardTitForTatPlayer_obs1_2026-04-14T13_46_.csv
Observer data saved to ipd_SlowTitForTatPlayer_obs1_2026-04-14T13_46_.csv
Observer data saved to ipd_TitFor2TatPlayer_obs1_2026-04-14T13_46_.csv
Observer data saved to ipd_GradualPlayer_obs1_2026-04-14T13_46_.csv
Observer data saved to ipd_PavlovPlayer_obs1_2026-04-14T13_46_.csv
Observer data saved to ipd_ProberPlayer_obs1_2026-04-14T13_46_.csv
Observer data saved to ipd_SoftMajorityPlayer_obs1_2026-04-14T13_47_.csv
Observer data saved to ipd_HardMajorityPlayer_obs1_2026-04-14T13_47_.csv
Observer data save

## Análise do Resultado

In [ ]:
# Configurações iniciais
sns.set(style="whitegrid")
pd.options.mode.chained_assignment = None  # Evita warnings desnecessários

# Config the paths
input_dir = Path("C:\\Users\\Nathan\\Home\\gEco\\ipd\\ipd\\runs")

output_dir = Path("C:\\Users\\Nathan\\Home\\gEco\\ipd\\ipd\\results")
output_dir.mkdir(exist_ok=True)

# CSV
csv_files = list(input_dir.glob("*.csv"))

# Verificando se existem arquivos CSV
if not csv_files:
    raise ValueError(f"Nenhum arquivo CSV foi encontrado em: {input_dir.resolve()}")

# Gerando o DataFrame consolidado
runs_df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)
runs_df_or = runs_df.copy()
runs_df = runs_df.dropna()

# Calculando a diferença de payoffs
runs_df["diff_payoffs"] = runs_df["my_payoff"] - runs_df["other_payoff"]

# Agrupando os dados para calcular médias e somas de payoffs
sum_runs = (
    runs_df.groupby(["scenario", "step", "strategy_name"])
    .agg(
        payoff_mean=("my_payoff", "mean"),
        payoff_sum=("my_payoff", "sum"),
    )
    .reset_index()
)

# Calculando cooperação e defecção
sum_cooperation = (
    runs_df.groupby(["scenario", "step"])
    .agg(
        my_play_C=("my_play", lambda x: (x == "C").sum()),
        my_play_D=("my_play", lambda x: (x == "D").sum()),
        ot_play_C=("other_play", lambda x: (x == "C").sum()),
        ot_play_D=("other_play", lambda x: (x == "D").sum()),
        players=("my_play", "count"),
    )
    .reset_index()
)

sum_cooperation["cooperation"] = sum_cooperation["my_play_C"] / sum_cooperation["players"]
sum_cooperation["defection"] = sum_cooperation["my_play_D"] / sum_cooperation["players"]

# Tabela com média e mediana por agente
agent_summary = (
    runs_df.groupby("strategy_name")
    .agg(
        payoff_mean=("my_payoff", "mean"),
        payoff_median=("my_payoff", "median"),
        payoff_sum=("my_payoff", "sum"),
        observations=("my_payoff", "count"),
    )
    .reset_index()
    .sort_values(by="payoff_mean", ascending=False)
)

# Exibindo a tabela
print("Tabela de média e mediana por agente:")
display(agent_summary)

# Salvando a tabela
agent_summary.to_csv(output_dir / "agent_summary.csv", index=False)

# Gera uma paleta mais saturada
unique_strategies = sum_runs["strategy_name"].unique()
palette = sns.color_palette("bright", n_colors=len(unique_strategies))

# Cria um mapeamento de cores
color_dict = dict(zip(unique_strategies, palette))

# Gráfico 1: Payoff médio por estratégia ao longo do tempo
g = sns.lmplot(
    data=sum_runs, 
    x="step", 
    y="payoff_mean",
    hue="strategy_name",
    palette=color_dict,
    scatter=True, 
    lowess=True, 
    line_kws={"alpha": 0.7},
    scatter_kws={"alpha": 0.1, "s": 10},
    height=6, aspect=1.5
)

# Títulos e eixos
g.set_axis_labels("Passo", "Payoff Médio")
g.fig.suptitle("Payoff Médio por Estratégia ao Longo do Tempo", fontsize=14, y=1.03)

# Remove a legenda original
if g._legend is not None:
    g._legend.remove()

# Cria legenda personalizada com patches coloridos
handles = [Patch(color=color_dict[strategy], label=strategy) for strategy in unique_strategies]
g.ax.legend(handles=handles, title="Estratégia", fontsize=10, title_fontsize=11)

# Salvar e mostrar
plt.savefig(output_dir / "payoff_mean_by_strategy.png", bbox_inches="tight")
plt.show()

# Gráfico 2: Distribuição de payoff médio por estratégia
plt.figure(figsize=(10, 6))
sns.violinplot(data=sum_runs, x="strategy_name", y="payoff_mean", inner=None)
sns.boxplot(data=sum_runs, x="strategy_name", y="payoff_mean", width=0.2, color="white")
sns.pointplot(
    data=sum_runs,
    x="strategy_name",
    y="payoff_mean",
    color="red",
    join=False,
    markers="o",
    scale=1.5,
)
plt.title("Distribuição de Payoff Médio por Estratégia")
plt.xlabel("Estratégia")
plt.ylabel("Payoff Médio")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(output_dir / "payoff_distribution_by_strategy.png")
plt.show()

# Gráfico 3A: Cooperação ao longo do tempo (dispersão)
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=sum_cooperation, 
    x="step", 
    y="cooperation", 
    alpha=0.5
)
plt.title("Cooperação ao Longo do Tempo")
plt.xlabel("Passo")
plt.ylabel("Cooperação")
plt.savefig(output_dir / "cooperation_scatter.png")
plt.show()

# Gráfico 3B: Cooperação ao longo do tempo com suavização
g = sns.lmplot(
    data=sum_cooperation, 
    x="step", 
    y="cooperation", 
    scatter=True, 
    lowess=True, 
    line_kws={"alpha": 0.5},
    scatter_kws={"alpha": 0.1, "s": 10},
    height=6, aspect=1.5
)

g.set_axis_labels("Passo", "Cooperação")
g.fig.suptitle("Cooperação ao Longo do Tempo", fontsize=14, y=1.03)

plt.savefig(output_dir / "cooperation_over_time.png", bbox_inches="tight")
plt.show()